In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="11A_4t23exGlVicNCqwc9V3sD-N7o9WUr", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/02_00_intro.mp3"))


In [ ]:
# 🔧 Setup: Run this cell first!
# Check GPU availability and install dependencies

import torch
import sys

# Check GPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    device = torch.device('cpu')
    print("⚠️ No GPU detected. Some cells may run slowly.")
    print("   Go to Runtime → Change runtime type → GPU")

print(f"\n📦 Python {sys.version.split()[0]}")
print(f"🔥 PyTorch {torch.__version__}")

# Set random seeds for reproducibility
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"🎲 Random seed set to {SEED}")

%matplotlib inline

In [ ]:
#@title 🎧 Listen: Overview Why It Matters
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_02_overview_why_it_matters.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


# RAGAS Evaluation Framework — Vizuara

# RAGAS Evaluation Framework
## Measuring Context Quality with Automated Metrics

In this notebook, we build the four core RAGAS metrics from scratch: Context Precision, Context Recall, Faithfulness, and Answer Relevancy.

**What you will learn:**
- The four RAGAS metrics and their mathematical formulations
- How to compute each metric from scratch in Python
- How to interpret results and diagnose pipeline weaknesses

## 1. Why Does This Matter?

You have built a RAG pipeline. But how do you know it is working? Without measurement, every change is a guess. RAGAS gives us four complementary metrics that answer: **is our context good enough?**

In [ ]:
#@title 🎧 Code Walkthrough: Install Imports
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_03_install_imports.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
!pip install -q numpy scikit-learn matplotlib

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
#@title 🎧 Listen: Building Intuition
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_04_building_intuition.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 2. Building Intuition

| Metric | Question It Answers | Failure It Catches |
|--------|--------------------|--------------------|
| **Context Precision** | Did we retrieve relevant chunks? | Noise in context |
| **Context Recall** | Did we retrieve ALL needed info? | Missing information |
| **Faithfulness** | Does the answer stick to the context? | Hallucination |
| **Answer Relevancy** | Does the answer address the question? | Off-topic responses |

In [ ]:
#@title 🎧 Listen: Context Precision Math
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_05_context_precision_math.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 3. The Mathematics

### 3.1 Context Precision

$$\text{Context Precision} = \frac{1}{|\text{relevant}|} \sum_{k=1}^{K} \mathbb{1}[\text{relevant}_k] \cdot \frac{\text{relevant in top } k}{k}$$

**Worked example:** Labels = [True, False, True, True, False]
- Pos 1: relevant, precision = 1/1 = 1.000
- Pos 3: relevant, precision = 2/3 = 0.667
- Pos 4: relevant, precision = 3/4 = 0.750
- CP = (1.000 + 0.667 + 0.750) / 3 = 0.806

In [ ]:
#@title 🎧 Code Walkthrough: Context Precision Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_06_context_precision_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def compute_context_precision(relevance_labels):
    precisions_at_relevant = []
    relevant_count = 0
    for k, is_relevant in enumerate(relevance_labels, start=1):
        if is_relevant:
            relevant_count += 1
            precision_at_k = relevant_count / k
            precisions_at_relevant.append(precision_at_k)
            print(f"  Position {k}: RELEVANT -> precision@{k} = {relevant_count}/{k} = {precision_at_k:.3f}")
        else:
            print(f"  Position {k}: not relevant (skip)")
    if not precisions_at_relevant:
        return 0.0
    avg_precision = sum(precisions_at_relevant) / len(precisions_at_relevant)
    return avg_precision

print("=== Context Precision Example ===\n")
labels = [True, False, True, True, False]
cp = compute_context_precision(labels)
print(f"\nContext Precision: {cp:.3f}")

In [ ]:
#@title 🎧 Code Walkthrough: Context Precision Compare
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_07_context_precision_compare.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Compare good vs bad retrieval ranking
print("=== Good Retrieval (relevant first) ===")
cp_good = compute_context_precision([True, True, True, False, False])
print(f"Precision: {cp_good:.3f}\n")

print("=== Bad Retrieval (relevant last) ===")
cp_bad = compute_context_precision([False, False, True, True, True])
print(f"Precision: {cp_bad:.3f}\n")

print(f"Difference: {cp_good - cp_bad:.3f} — ranking matters!")

In [ ]:
#@title 🎧 Listen: Faithfulness Math
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_08_faithfulness_math.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### 3.2 Faithfulness

$$\text{Faithfulness} = \frac{|\text{claims supported by context}|}{|\text{total claims in answer}|}$$

In [ ]:
#@title 🎧 Code Walkthrough: Faithfulness Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_09_faithfulness_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def compute_faithfulness(answer_claims, context, threshold=0.3):
    if not answer_claims:
        return 0.0
    vectorizer = TfidfVectorizer(stop_words='english')
    supported = 0
    context_sentences = [s.strip() for s in context.split('. ') if len(s.strip()) > 5]
    for claim in answer_claims:
        try:
            texts = [claim] + context_sentences
            tfidf = vectorizer.fit_transform(texts)
            similarities = cosine_similarity(tfidf[0:1], tfidf[1:]).flatten()
            max_sim = similarities.max()
            status = "SUPPORTED" if max_sim >= threshold else "NOT SUPPORTED"
            print(f"  [{max_sim:.3f}] {status}: \"{claim[:70]}\"")
            if max_sim >= threshold:
                supported += 1
        except ValueError:
            pass
    faithfulness = supported / len(answer_claims)
    print(f"\n  Supported: {supported}/{len(answer_claims)} = {faithfulness:.3f}")
    return faithfulness

print("=== Faithfulness Example ===\n")
context = "BERT was published by Google in 2018. It uses bidirectional self-attention. The model was pre-trained on BooksCorpus and English Wikipedia."
claims = [
    "BERT was published by Google in 2018",
    "BERT uses bidirectional self-attention",
    "BERT has 340 million parameters",
    "BERT outperforms GPT-4 on reasoning"
]
faith = compute_faithfulness(claims, context)
print(f"\nFaithfulness: {faith:.3f}, Hallucination rate: {1-faith:.3f}")

In [ ]:
#@title 🎧 Listen: Answer Relevancy
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_10_answer_relevancy.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### 3.3 Answer Relevancy

In [ ]:
def compute_answer_relevancy(question, answer):
    vectorizer = TfidfVectorizer(stop_words='english')
    tfidf = vectorizer.fit_transform([question, answer])
    return cosine_similarity(tfidf[0:1], tfidf[1:2])[0][0]

question = "How does BERT handle bidirectional context?"
good_answer = "BERT processes text bidirectionally using masked language modeling and self-attention."
bad_answer = "Transformers use encoder and decoder stacks with multi-head attention."

print(f"Good answer relevancy: {compute_answer_relevancy(question, good_answer):.3f}")
print(f"Bad answer relevancy:  {compute_answer_relevancy(question, bad_answer):.3f}")

In [ ]:
#@title 🎧 Code Walkthrough: Ragas Evaluator Class
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_11_ragas_evaluator_class.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 4. Complete RAGAS Evaluator

In [ ]:
class RAGASEvaluator:
    def __init__(self, threshold=0.3):
        self.threshold = threshold

    def evaluate(self, question, context_chunks, relevance_labels,
                 answer, answer_claims):
        full_context = ' '.join(context_chunks)
        metrics = {
            'context_precision': compute_context_precision(relevance_labels),
            'faithfulness': compute_faithfulness(answer_claims, full_context, self.threshold),
            'answer_relevancy': compute_answer_relevancy(question, answer),
        }
        valid = [v for v in metrics.values() if v > 0]
        metrics['ragas_score'] = len(valid) / sum(1/s for s in valid) if valid else 0
        return metrics

    def print_report(self, metrics):
        print("\n" + "=" * 50)
        print("         RAGAS EVALUATION REPORT")
        print("=" * 50)
        for metric, value in metrics.items():
            bar = "#" * int(value * 30) + "." * (30 - int(value * 30))
            quality = "Excellent" if value > 0.85 else "Good" if value > 0.7 else "Needs Work"
            print(f"  {metric:<22} {value:.3f} |{bar}| {quality}")
        print("=" * 50)

evaluator = RAGASEvaluator()
print("Running RAGAS evaluation...\n")
metrics = evaluator.evaluate(
    question="How does BERT process bidirectional context?",
    context_chunks=[
        "BERT uses masked language modeling for bidirectional representation.",
        "The weather in San Francisco is often foggy.",
        "BERT's self-attention allows each token to attend to all tokens.",
        "Bidirectional context is key to BERT's strong NLP performance.",
        "GPT uses causal attention which is unidirectional."
    ],
    relevance_labels=[True, False, True, True, False],
    answer="BERT processes text bidirectionally using masked language modeling and self-attention.",
    answer_claims=["BERT processes text bidirectionally", "BERT uses masked language modeling",
                   "BERT uses self-attention", "Each token attends to all other tokens"]
)
evaluator.print_report(metrics)

In [ ]:
#@title 🎧 Before You Start: Todo1 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_12_todo1_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 5. Your Turn: Diagnose a Broken Pipeline

**TODO Exercise 1:** Run the evaluation below, identify which metric is lowest, and explain why.

In [ ]:
#@title 🎧 Before You Start: Todo1 Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_13_todo1_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
broken_metrics = evaluator.evaluate(
    question="What is the training data used for GPT-3?",
    context_chunks=[
        "Common Crawl is a large web scraping dataset.",
        "WebText2 is curated web text for language model training.",
        "BookCorpus contains text from 11,000 unpublished books.",
        "Wikipedia dumps are used for pre-training language models.",
        "ImageNet is designed for image recognition research."
    ],
    relevance_labels=[True, True, True, True, False],
    answer="GPT-3 was trained on Common Crawl, WebText2, BookCorpus, and Wikipedia. It has 175B parameters and cost $12M to train.",
    answer_claims=[
        "GPT-3 trained on Common Crawl", "GPT-3 trained on WebText2",
        "GPT-3 has 175B parameters",  # not in context
        "Training cost $12 million"    # hallucinated
    ]
)
evaluator.print_report(broken_metrics)

# TODO: Which metric is lowest? Why? What would you fix?

In [ ]:
#@title 🎧 Before You Start: Todo2 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_14_todo2_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


**TODO Exercise 2:** Write a function that reranks chunks by relevance to maximize context precision.

In [ ]:
#@title 🎧 Before You Start: Todo2 Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_15_todo2_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def optimize_chunk_ranking(chunks, query):
    """
    TODO: Rerank chunks by TF-IDF similarity to query.
    1. Compute similarity between each chunk and query
    2. Sort by similarity (highest first)
    3. Return reranked list
    """
    # YOUR CODE HERE
    return chunks

In [ ]:
#@title 🎧 Transition: Aggregated Eval Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_16_aggregated_eval_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 6. Aggregated Evaluation Across Multiple Queries

In [ ]:
#@title 🎧 What to Look For: Aggregated Eval Viz
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_17_aggregated_eval_viz.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
import matplotlib.pyplot as plt

test_set = [
    {"question": "What optimizer does BERT use?",
     "chunks": ["BERT uses Adam optimizer with warmup.", "BERT has 110M parameters."],
     "labels": [True, False],
     "answer": "BERT uses Adam optimizer with warmup schedule.",
     "claims": ["BERT uses Adam optimizer", "BERT uses warmup schedule"]},
    {"question": "How many layers does GPT-2 have?",
     "chunks": ["GPT-2 Large has 36 transformer layers.", "GPT-2 was released by OpenAI."],
     "labels": [True, True],
     "answer": "GPT-2 Large has 36 layers, developed by OpenAI.",
     "claims": ["GPT-2 has 36 layers", "Developed by OpenAI"]},
]

all_metrics = [evaluator.evaluate(t["question"], t["chunks"], t["labels"],
                                   t["answer"], t["claims"]) for t in test_set]

metric_names = ['context_precision', 'faithfulness', 'answer_relevancy', 'ragas_score']
avg = {n: np.mean([m[n] for m in all_metrics]) for n in metric_names}

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#4285F4', '#EA4335', '#34A853', '#FBBC05']
bars = ax.barh(metric_names, [avg[n] for n in metric_names], color=colors)
ax.set_xlim(0, 1)
ax.set_xlabel('Score')
ax.set_title('RAGAS Evaluation Results (Averaged)')
for bar, val in zip(bars, [avg[n] for n in metric_names]):
    ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2, f'{val:.3f}', va='center')
plt.tight_layout()
plt.savefig('ragas_results.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
#@title 🎧 Wrap-Up: Closing
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_18_closing.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 7. Reflection and Next Steps

**Key takeaways:**
- Each RAGAS metric catches a different failure mode
- Faithfulness is the most critical for production (catches hallucination)
- Ranking matters as much as selection for context precision

**Next notebook:** Human evaluation protocols with Cohen's Kappa.